# core

> Helper functions for textplumber.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import os

In [ ]:
#| export
def pass_tokens(tokens:list 
				) -> list: 
	""" Pass through function so pre-tokenized input can be passed to CountVectorizer or TfidfVectorizer. """
	return tokens

In [ ]:
#| export
def get_stop_words(save_to:str|None = 'lexicons_stop_words.txt' # where to save the file, None will not save
					):
	""" Get stop words from NLTK (with option to cache to disk). """
	if save_to is not None:
		if os.path.exists(save_to):
			with open(save_to, 'r', encoding='utf-8') as f:
				stop_words = f.read().splitlines()
				return stop_words
		else:
			save_path = os.path.dirname(save_to)
			if save_path != '' and not os.path.exists(save_path):
				os.makedirs(save_path)

	import nltk
	from nltk.corpus import stopwords
	nltk.download('stopwords')
	stop_words = stopwords.words('english')

	if save_to is not None:
		with open(save_to, 'w', encoding='utf-8') as f:
			for word in stopwords.words('english'):
				f.write(word + '\n')

	return stop_words


In [ ]:
#| hide
if os.path.exists('lexicons_stop_words.txt'):
	os.remove('lexicons_stop_words.txt')
stop_words = get_stop_words()
assert os.path.exists('lexicons_stop_words.txt')
# checking fresh load
assert stop_words[:3] == ['a', 'about', 'above']
# checking load from cache
stop_words = get_stop_words()
assert stop_words[:3] == ['a', 'about', 'above']
os.remove('lexicons_stop_words.txt')

[nltk_data] Downloading package stopwords to /home/geoff/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#| export
def get_example_data(
		train_split_name:str = 'train', # this can be defined, but probably unnecessary to change
		test_split_name:str = 'validation', # could be 'test'
		label_column:str = 'category', # 'category' or 'style'
		target_labels:list = ['blog', 'author', 'speech'] # see the dataset card for information on labels https://huggingface.co/datasets/hallisky/AuthorMix
		):
	""" Get data for examples using Huggingface dataset hallisky/AuthorMix. Majority classes are automatically undersampled. """

	import numpy as np
	from datasets import load_dataset, ClassLabel
	from imblearn.under_sampling import RandomUnderSampler

	text_column = 'text'

	dataset = load_dataset('hallisky/AuthorMix')

	class_feature = ClassLabel(names=dataset['train'].unique(label_column))
	for split in dataset.keys():
		dataset[split] = dataset[split].cast_column(label_column, class_feature)

	label_names = dataset['train'].features[label_column].names

	target_classes = [label_names.index(name) for name in target_labels]
	target_names = [label_names[i] for i in target_classes]

	X_train = np.array(dataset[train_split_name][text_column])
	y_train = np.array(dataset[train_split_name][label_column])
	X_test = np.array(dataset[test_split_name][text_column])
	y_test = np.array(dataset[test_split_name][label_column])

	mask = np.isin(y_train, target_classes)
	mask_test = np.isin(y_test, target_classes)

	X_train = X_train[mask]
	y_train = y_train[mask]
	X_test = X_test[mask_test]
	y_test = y_test[mask_test]

	# undersampling all but the minority class to balance the training data
	X_train = X_train.reshape(-1, 1)
	X_train, y_train = RandomUnderSampler(random_state=0).fit_resample(X_train, y_train)
	X_train = X_train.reshape(-1)

	return X_train, y_train, X_test, y_test, target_classes, target_names


The [AuthorMix dataset](https://huggingface.co/datasets/hallisky/AuthorMix) is used for testing and for examples to document specific components. There are two fields that can be used as label columns. The default is 'category', but 'style' is also available. Here are some possible configurations. 

In [ ]:
from textplumber.report import preview_splits

In [ ]:
print('Load all classes ...')
X_train, y_train, X_test, y_test, target_classes, target_names = get_example_data()
preview_splits(X_train, y_train, X_test, y_test, target_names)

print('With specific target labels ...')
X_train, y_train, X_test, y_test, target_classes, target_names = get_example_data(target_labels = ['author', 'speech'])
preview_splits(X_train, y_train, X_test, y_test, target_names)

print('Using style rather than category with author names as labels ...')
X_train, y_train, X_test, y_test, target_classes, target_names = get_example_data(label_column = 'style', target_labels = ['fitzgerald', 'hemingway', 'woolf'])
preview_splits(X_train, y_train, X_test, y_test, target_names)

print('Using style rather than category with president names as labels ...')
X_train, y_train, X_test, y_test, target_classes, target_names = get_example_data(label_column = 'style', target_labels = ['obama', 'trump'])
preview_splits(X_train, y_train, X_test, y_test, target_names)

Load all classes ...
Train: 9444 samples, 3 classes


,label_name,count
0,,
0,blog,3148
1,author,3148
2,speech,3148


Test: 3598 samples, 3 classes


,label_name,count
0,,
1,blog,1877
2,author,981
0,speech,740


With specific target labels ...
Train: 6296 samples, 2 classes


...

,label_name,count
0,,
4,fitzgerald,885
5,hemingway,504
3,woolf,488


Using style rather than category with president names as labels ...
Train: 2336 samples, 2 classes


,label_name,count
0,,
0,obama,1168
2,trump,1168


Test: 601 samples, 2 classes


,label_name,count
0,,
2,obama,328
0,trump,273


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()